In [8]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

In [9]:
df = pd.read_csv("gene_expression.csv")
df.head()

,Gene One,Gene Two,Cancer Present
0,4.3,3.9,1
1,2.5,6.3,0
2,5.7,3.9,1
3,6.1,6.2,0
4,7.4,3.4,1


In [10]:
X = df.iloc[:,:-1]
y = df.iloc[:, -1]

In [11]:
# fig, ax = plt.subplots(figsize=(8, 5.5))
# palette = ["#5b83de", "#20a475", "#dd7852"]
# for class_id, name in enumerate(X.target_names):
#     subset = X[y == class_id]
#     ax.scatter(subset.iloc[:, 0], subset.iloc[:, 1], label=name,
#                color=palette[class_id], alpha=0.85, s=43, edgecolor="white", linewidth=0.4)
# ax.set(xlabel=feature_names[0], ylabel=feature_names[1], title="Iris: two-feature view of three classes")
# ax.legend(title="Species")
# plt.show()

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print("Train: ", X_train.shape[0])
print("Test: ", X_test.shape[0])

Train:  2250
Test:  750


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

print(knn_pipeline)

Pipeline(steps=[('scaler', StandardScaler()), ('knn', KNeighborsClassifier())])


In [15]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle = True, random_state=42)
baseline_cv = cross_validate(
    knn_pipeline,
    X_train, y_train,
    cv=cv,
    scoring="accuracy",
    return_train_score=True,
)

print("Baseline validation accuracy per fold:", np.round(baseline_cv["test_score"], 3))
print(f"Mean validation accuracy: {baseline_cv['test_score'].mean():.3f}")
print(f"Std across folds:        {baseline_cv['test_score'].std():.3f}")
print(f"Mean train accuracy:     {baseline_cv['train_score'].mean():.3f}")

Baseline validation accuracy per fold: [0.929 0.92  0.931 0.922 0.938]
Mean validation accuracy: 0.928
Std across folds:        0.006
Mean train accuracy:     0.946


In [16]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "knn__n_neighbors": list(range(1,22,2)),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1,2],
}

grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    return_train_score=True,
    refit=True,
    n_jobs=1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print(f"Best mean CV accuracy: {grid.best_score_:.3f}")
print("Best estimator:", grid.best_estimator_)

Best parameters: {'knn__n_neighbors': 21, 'knn__p': 1, 'knn__weights': 'uniform'}
Best mean CV accuracy: 0.934
Best estimator: Pipeline(steps=[('scaler', StandardScaler()),
                ('knn', KNeighborsClassifier(n_neighbors=21, p=1))])
